In [ ]:
# --- project bootstrap ---
import os
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
os.chdir(ROOT)
# -------------------------


In [31]:
# %% Imports & hot-reload
import os, importlib, numpy as np
from IPython.display import Image, display

import fgtn.classA_U1FGTN as mod
importlib.reload(mod)
from fgtn.classA_U1FGTN import classA_U1FGTN
# small helpers
def show(path, width=700):
    if os.path.isfile(path):
        display(Image(filename=path, width=width))
    else:
        print(f"[missing file] {path}")

def list_dir(d):
    if os.path.isdir(d):
        for f in sorted(os.listdir(d)):
            print(f"- {f}")
    else:
        print(f"[missing dir] {d}")

In [8]:
# %% Parameters
Nx, Ny = 21, 21
cycles = 20            # number of RAC cycles (time steps for this model)
samples = 10          # number of trajectories to average over
nshell  = 5        # truncation for OW (None = no truncation)
DW = True             # domain-wall geometry
filling_frac = 0.5    # for default G0 construction

# Compute-setup for (re)generation if cache is missing:
backend = "loky"      # or "threading"
n_jobs  = None        # None -> auto (min(samples, CPU))

In [ ]:
# %% Build/Load histories for init_kind="default"
m_def = classA_U1FGTN(
    Nx, Ny, DW=DW, cycles=cycles, samples=samples, nshell=nshell,
    filling_frac=filling_frac, init_kind="default", prompt_on_miss=True, load_top_only=True
)

# If cache is missing, generate and save:
_ = m_def.ensure_G_history_samples(
    samples=samples, cycles=cycles, n_jobs=n_jobs, backend=backend, init_kind="default"
)

# Quick peek
bundle_def = m_def.get_history_bundle()
print("DEFAULT -> S, T, Nlayer:", bundle_def["samples"], bundle_def["T"], bundle_def["G_hist"].shape[-1])

DWs at x=(5, 14)
------------------------- classA_U1FGTN Initialized -------------------------
DEFAULT -> S, T, Nlayer: 10 20 882


In [ ]:
# %% Build/Load histories for init_kind="maxmix_top"
m_mm = classA_U1FGTN(
    Nx, Ny, DW=DW, cycles=cycles, samples=samples, nshell=nshell,
    filling_frac=filling_frac, init_kind="maxmix_top", prompt_on_miss=False, load_top_only=True
)

_ = m_mm.ensure_G_history_samples(
    samples=samples, cycles=cycles, n_jobs=n_jobs, backend=backend, init_kind="maxmix_top"
)

bundle_mm = m_mm.get_history_bundle()
print("MAXMIX_TOP -> S, T, Nlayer:", bundle_mm["samples"], bundle_mm["T"], bundle_mm["G_hist"].shape[-1])

In [ ]:
# %% Real-space Chern history (init_kind="default")
# Single-trajectory (first sample)
fig1, ax1, ch_single = m_def.plot_real_space_chern_history(
    filename="figs/chern_history/chern_history_single_default.pdf",
    traj_avg=False
)
print("Saved:", "figs/chern_history/chern_history_single_default.pdf")

# Trajectory-averaged vs marker-of-average (two panels)
fig2, (axL, axR), (C_traj_res, C_traj_avg) = m_def.plot_real_space_chern_history(
    filename="figs/chern_history/chern_history_avg_default.pdf",
    traj_avg=True
)
print("Saved:", "figs/chern_history/chern_history_avg_default.pdf")

In [ ]:
# %% Chern marker animations (init_kind="default")
gif_path, final_path, *_ = m_def.chern_marker_dynamics(
    outbasename=f"chern_marker_N{Nx}x{Ny}_C{cycles}_S{samples}_default",
    vmin=-1.0, vmax=1.0, cmap='RdBu_r', traj_avg=True  # try False for single-trajectory version
)
print("GIF:", gif_path)
print("Final frame:", final_path)

show(final_path, width=600)

In [30]:
# %% Correlation y-profiles (traj-resolved & avg) + spectra + Chern maps
pdf_path = m_def.plot_corr_y_profiles_v2(save=False)



TypeError: classA_U1FGTN.plot_corr_y_profiles_v2() got an unexpected keyword argument 'save'

In [ ]:
Gtt_history_samples = m_def.get_top_histories()
print(len(m_def.G_history_samples[0][0])) 
print(len(Gtt_history_samples[0][0]))
print((2*2*Nx*Ny))
m_def.plot_co

1764
882
1764


In [ ]:
# %% Entanglement contour suite (requires init_kind="maxmix_top")
results = m_mm.entanglement_contour_suite(
    filename_profiles=None, filename_prefix_dyn=None
)
print(results)

# Preview final PNG and GIFs if they exist
show(results["profiles_pdf"], width=800)
show(results["dyn_final_png"], width=800)  # if present in your function's return
show(results["dyn_gif_traj"], width=400)
show(results["dyn_gif_avg"],  width=400)

In [ ]:
# %% Sanity: eig spectra of final top-layer G (default vs maxmix_top)
import matplotlib.pyplot as plt
G_def_final = m_def.current_final_G(averaged=True)    # average over samples
G_mm_final  = m_mm.current_final_G(averaged=True)

ev_def = np.linalg.eigvalsh(np.asarray(G_def_final))
ev_mm  = np.linalg.eigvalsh(np.asarray(G_mm_final))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
ax1.plot(np.sort(ev_def), '.', ms=3); ax1.set_title("eigvals(G_final) — default")
ax2.plot(np.sort(ev_mm),  '.', ms=3); ax2.set_title("eigvals(G_final) — maxmix_top")
for ax in (ax1, ax2):
    ax.set_xlabel("index"); ax.set_ylabel("eigenvalue"); ax.grid(True, alpha=0.3)
plt.show()